# Projecting Aggression network into new mice

In [ ]:
import numpy as np
import sklearn.decomposition as dp
import pickle
import sys,os
import numpy.random as rand
from sklearn.linear_model import LogisticRegression as LR
from sklearn.metrics import auc,roc_curve,roc_auc_score
from sklearn.utils.random import sample_without_replacement
from scipy import io
import pandas as pd
import matplotlib.pyplot as plt

## Load in the labels and backprojection

In [ ]:
labelDict = io.loadmat('Windows.mat')
mouse = np.squeeze(labelDict['mouse'])
group = np.squeeze(labelDict['group'])
time = np.squeeze(labelDict['time'])
condition = np.squeeze(labelDict['condition'])
behavior = np.squeeze(labelDict['behavior'])

In [ ]:
pDict = io.loadmat('CL_baseline_projection2.mat')
Scores_supervised = np.squeeze(pDict['Scores_supervised'])
Scores_unsupervised = pDict['Scores_unsupervised']

## Set up the labels Y as task (Condition=4,Behavior=1) vs all

In [ ]:
y = np.zeros(len(mouse))
idx_pos = (condition==4)&(behavior==1)
y[idx_pos] = 1

## Actually go through and compute the AUCs by mouse for the supervised factor

In [ ]:
mice = np.unique(mouse)
nMice = len(mice)

In [ ]:
auc_sups = np.zeros(nMice)
for i in range(nMice):
    idx_mouse = mouse==mice[i]
    y_true = y[idx_mouse]
    y_hat = Scores_supervised[idx_mouse]
    auc_sups[i] = roc_auc_score(y_true,y_hat)

In [ ]:
for i in range(nMice):
    print('%s AUC: %0.2f'%(mice[i],auc_sups[i]))
mean = np.mean(auc_sups)
ci = np.std(auc_sups)/np.sqrt(nMice)*1.96
print('Test set AUC confidence interval: %0.2f +- %0.3f'%(mean,ci))

## Now lets look at the unsupervised factors

In [ ]:
auc_unsup = np.zeros((nMice,7))
for i in range(nMice):
    for j in range(7):
        idx_mouse = mouse==mice[i]
        y_true = y[idx_mouse]
        y_hat = Scores_unsupervised[idx_mouse,j]
        auc_unsup[i,j] = roc_auc_score(y_true,y_hat)
means = np.mean(auc_unsup,axis=0)
stds = np.std(auc_unsup,axis=0)
ci = 1.96*stds/np.sqrt(nMice)

In [ ]:
auc_unsup[:,0]

In [ ]:
for j in range(7):
    mainStr = 'Test set AUC CI Factor %d: %0.2f +- %0.2f'%(j+1,means[j],ci[j])
    print(mainStr)

## Finally get the correlation between the factors

In [ ]:
S_tot = np.zeros((len(mouse),8))
S_tot[:,0] = Scores_supervised
S_tot[:,1:] = Scores_unsupervised
cc = np.corrcoef(S_tot.T)

## Correlation between the most positive factor and supervised factor

In [ ]:
print('Correlation is',cc[0,1])

## Add in the remaining 3 mice

In [ ]:
mice_new = np.zeros(11).astype('U256')
for i in range(8):
    mice_new[i] = mice[i][0]
mice_new[8] = 'Mouse048'
mice_new[9] = 'Mouse7980'
mice_new[10] = 'Mouse7998'
print(mice_new)

In [ ]:
first_three = np.genfromtxt('Supervised3_tot_0.csv',delimiter=',')

In [ ]:
aucs_new = np.zeros(11)
aucs_new[:8] = auc_sups
aucs_new[8] = first_three[0,0]
aucs_new[9] = first_three[0,1]
aucs_new[10] = first_three[0,2]

In [ ]:
unsupervised = np.vstack((auc_unsup,first_three[1:].T))

In [ ]:
mean = np.mean(aucs_new)
ci = np.std(aucs_new)/np.sqrt(len(aucs_new))*1.96
print('Test set AUC confidence interval: %0.2f +- %0.3f'%(mean,ci))

In [ ]:
means = np.mean(unsupervised,axis=0)
stds = np.std(unsupervised,axis=0)
ci = 1.96*stds/np.sqrt(nMice)
for j in range(7):
    mainStr = 'Test set AUC CI Factor %d: %0.2f +- %0.2f'%(j+1,means[j],ci[j])
    print(mainStr)

## Lets save things

In [ ]:
d = {'Mouse':mice_new,'Supervised AUC':aucs_new,'Unsupervised 1':unsupervised[:,0],
    'Unsupervised 2':unsupervised[:,1],'Unsupervised 3':unsupervised[:,2],
    'Unsupervised 4':unsupervised[:,3],'Unsupervised 5':unsupervised[:,4],
    'Unsupervised 6':unsupervised[:,5],'Unsupervised 7':unsupervised[:,6],}
df = pd.DataFrame(data=d)
print(df)

In [ ]:
df.to_csv("Projection2.csv",index=False,sep=',')